# RAPID hydrograph review

Review observed gauge events, refreshed model baselines and individual collapse responses. This notebook diagnoses timing/shape and domain changes; predictor benchmarking remains separate. Updated 5 October 2026.

**Updated source:** active published Cunge registries under `network_variants/output`, including gauge overlays, parent/child plots and timing tables. No historical-refresh fallback. Peak times are native UTC. Earlier resistance values are not transferred across releases. Maps highlight saved domain membership only after validating its topology against current parent/child graphs; ZOOM_TO_DOMAIN=True zooms to those links. The commented legacy investigation cells below require a current-domain adapter before reactivation.

## Purpose and reading order

The original notebook only inspected zero/positive/negative collapse timing groups and their selected resistance domains. It did not validate against gauges and used historical runs. This version reads the October hydraulic refresh, adds observed/model overlays on verified actual UTC, and preserves those three inspection groups. A sampled zero means a timestamp tie, not an identical hydrograph. Interpolated timing is shown as a separate diagnostic; no per-river calibration or peak alignment is applied.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate
    raise FileNotFoundError("Could not locate the river-hierarchy repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
OUTPUT_ROOT = REPO_ROOT / "network_variants" / "output"
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))

import importlib
import rapid_tools.current_review as current_review
importlib.reload(current_review)
from rapid_tools.current_review import (
    build_current_analysis_frame, gauge_overlay, select_network_pairs,
    parent_child_hydrograph_figure, parent_child_network_figure,
    collapse_hydrograph_overlay,
)
import plotly.graph_objects as go
import json


## Configuration and input assembly

The cells below load the completed experiments and form the single analysis population used by every investigation section. Run them before using the review cells.


In [2]:
EXAMPLE_IDS = ["02","03","05","11","13","15","21","22","37","41","45","46","48","54","55","56"]
# Read current published registries; unavailable examples are reported, never loaded from an older release.
PRIMARY_Y_METRIC = "lag_to_inflow_peak_pct_shift_vs_base"
# This percentage uses sampled peak lag; reviewed_feature_shift_minutes separately retains 54's centre.
# Historical resistance/domain metrics are not mixed into current timing results.


## Build the validated analysis population

Only states with the required RAPID and selected-domain equivalent-resistance records enter `plot_df`. The following cell is the shared input contract for all three response classes.


In [3]:
analysis_df = build_current_analysis_frame(EXAMPLE_IDS, output_root=OUTPUT_ROOT)
analysis_df["primary_pair_complete"] = np.isfinite(analysis_df[PRIMARY_Y_METRIC])
coverage = analysis_df.groupby(["example_id", "mode"]).agg(
    states=("state_id", "nunique"), unique_hydrographs=("fingerprint", "nunique"),
    timing_available=(PRIMARY_Y_METRIC, lambda s: np.isfinite(s).sum()),
).reset_index()
display(coverage)
print("Source: active published Cunge rapid_run_registry.csv; historical refresh is not loaded.")
print("Unique child comparisons:", len(analysis_df.loc[~analysis_df.technical_alias]))
display(analysis_df[["example_id","mode","state_id","sampled_peak_shift_minutes",
    "reviewed_feature_shift_minutes","peak_total_change_pct","parent_peak_utc","child_peak_utc",
    "registry_path"]].head())


/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:24: UserWarning: 11/indep: registry has historical schema; excluded from current review.
  warnings.warn(f'{ex}/{family}: registry has historical schema; excluded from current review.')
/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:24: UserWarning: 11/compound_bubbles: registry has historical schema; excluded from current review.
  warnings.warn(f'{ex}/{family}: registry has historical schema; excluded from current review.')
/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py:30: UserWarning: 11: no current published registry; excluded (no historical fallback).
  warnings.warn(f'{ex}: no current published registry; excluded (no historical fallback).')


,example_id,mode,states,unique_hydrographs,timing_available
0,02,compound,5,5,5
1,02,indep,26,26,26
2,02,island,26,26,26
3,03,compound,7,7,7
4,03,indep,31,31,31
5,03,island,31,31,31
6,05,compound,8,8,8
7,05,indep,43,43,43
8,05,island,43,43,43
9,13,compound,10,10,10


Source: active published Cunge rapid_run_registry.csv; historical refresh is not loaded.
Unique child comparisons: 2172


,example_id,mode,state_id,sampled_peak_shift_minutes,reviewed_feature_shift_minutes,peak_total_change_pct,parent_peak_utc,child_peak_utc,registry_path
0,02,indep,S001_unit_2,-7.0,-7.0,-0.008356,2025-08-01T22:59:30+00:00,2025-08-01T22:52:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
1,02,indep,S002_unit_18,1.0,1.0,0.029712,2025-08-01T22:59:30+00:00,2025-08-01T23:00:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
2,02,indep,S003_unit_17,3.0,3.0,0.373322,2025-08-01T22:59:30+00:00,2025-08-01T23:02:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
3,02,indep,S004_unit_5,0.0,0.0,-0.048360,2025-08-01T22:59:30+00:00,2025-08-01T22:59:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...
4,02,indep,S005_unit_25,-1.0,-1.0,-0.002408,2025-08-01T22:59:30+00:00,2025-08-01T22:58:30+00:00,/Users/6256481/Code/river-hierarchy/network_va...


In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# ── Settings ───────────────────────────────────────────────────────────
TIMING_COLUMN = "sampled_peak_shift_minutes"
# Use "reviewed_feature_shift_minutes" to use the centre timing for 54.

N_BINS = 80
X_RANGE = None           # Example: [-30, 30] to zoom; calculations use all data.
KDE_BANDWIDTH = "scott"

GROUPS = [
    "Simple",
    "Nested",
    "Compound member",
    "Island",
    "Full compound",
]

COLORS = {
    "Simple": "#2878B5",
    "Nested": "#9467BD",
    "Compound member": "#D95F02",
    "Island": "#2CA02C",
    "Full compound": "#D62728",
}

# ── Current responses: remove technical aliases ────────────────────────
df = analysis_df.loc[~analysis_df["technical_alias"]].copy()
df["example_id"] = df["example_id"].astype(str).str.zfill(2)

# Audited hierarchy membership for this release.
membership = pd.read_parquet(
    REPO_ROOT
    / "RAPID/outputs/geometry_response_foundation_20261012T120140"
    / "canonical_pairs.parquet"
).rename(columns={"example": "example_id"})

membership["example_id"] = (
    membership["example_id"].astype(str).str.zfill(2)
)

membership_columns = ["any_nested_unit", "any_compound_member"]

# Allow this cell to run even if membership columns were added previously.
df = df.drop(columns=membership_columns, errors="ignore").merge(
    membership[["example_id", "fingerprint", *membership_columns]],
    on=["example_id", "fingerprint"],
    how="left",
    validate="one_to_one",
)

if df[membership_columns].isna().any().any():
    raise ValueError(
        "Some current states lack audited hierarchy membership. "
        "Check that the membership snapshot matches the current release."
    )

# Mutually exclusive classification:
# independent island cases → nested → compound member → simple.
# Island aliases of independent units have already been removed.
# Mutually exclusive groups. Full-compound cases take priority.
df["group"] = np.select(
    [
        df["mode"].eq("compound"),
        df["mode"].eq("island"),
        df["any_nested_unit"].eq(True),
        df["any_compound_member"].eq(True),
    ],
    ["Full compound", "Island", "Nested", "Compound member"],
    default="Simple",
)

df["shift_min"] = pd.to_numeric(df[TIMING_COLUMN], errors="coerce")
valid = np.isfinite(df["shift_min"])
print(f"Missing/nonfinite timings excluded: {(~valid).sum()}")
df = df.loc[valid].copy()

if df.empty:
    raise ValueError("No valid peak shifts remain.")

# ── Shared bins and KDE evaluation grid ────────────────────────────────
lo = float(df["shift_min"].min())
hi = float(df["shift_min"].max())

if lo == hi:
    lo, hi = lo - 1, hi + 1

bin_width = (hi - lo) / N_BINS
grid = np.linspace(lo, hi, 3000)

fig = go.Figure()

for group in GROUPS:
    values = df.loc[df["group"].eq(group), "shift_min"].to_numpy()
    if not len(values):
        continue

    label = f"{group} (n={len(values)})"
    has_kde = len(values) > 1 and np.std(values) > 0

    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=lo, end=hi, size=bin_width),
        histnorm="probability density",
        marker_color=COLORS[group],
        opacity=0.18,
        name=label,
        legendgroup=group,
        showlegend=not has_kde,
        hovertemplate=(
            f"{group}<br>"
            "Shift bin: %{x} min<br>"
            "Density: %{y:.4g}<extra></extra>"
        ),
    ))

    if has_kde:
        kde = gaussian_kde(values, bw_method=KDE_BANDWIDTH)

        fig.add_trace(go.Scatter(
            x=grid,
            y=kde(grid),
            mode="lines",
            line=dict(color=COLORS[group], width=2.5),
            name=label,
            legendgroup=group,
            hovertemplate=(
                f"{group}<br>"
                "Shift: %{x:.2f} min<br>"
                "KDE density: %{y:.4g}<extra></extra>"
            ),
        ))

fig.add_vline(
    x=0,
    line_color="black",
    line_dash="dash",
    line_width=1,
)

timing_label = (
    "Sampled peak"
    if TIMING_COLUMN == "sampled_peak_shift_minutes"
    else "Reviewed crest feature"
)

fig.update_layout(
    title=f"{timing_label} timing changes by collapse group",
    template="plotly_white",
    barmode="overlay",
    width=1050,
    height=550,
    xaxis_title="Child − parent timing (minutes): earlier ← 0 → later",
    yaxis_title="Probability density (per minute)",
    legend=dict(
        title="Collapse group",
        groupclick="togglegroup",
    ),
    hovermode="closest",
)

if X_RANGE is not None:
    fig.update_xaxes(range=X_RANGE)

fig.show()

# ── Counts and timing summary ──────────────────────────────────────────
summary = (
    df.groupby("group")["shift_min"]
      .agg(
          n="size",
          earlier=lambda x: (x < 0).sum(),
          unchanged=lambda x: (x == 0).sum(),
          later=lambda x: (x > 0).sum(),
          median_minutes="median",
          minimum_minutes="min",
          maximum_minutes="max",
      )
      .reindex(GROUPS)
      .dropna(subset=["n"])
)

display(summary)

Missing/nonfinite timings excluded: 0


,n,earlier,unchanged,later,median_minutes,minimum_minutes,maximum_minutes
group,,,,,,,
Simple,258,241,8,9,-6.0,-47.0,6.0
Nested,738,537,148,53,-1.0,-137.0,77.0
Compound member,750,564,120,66,-2.0,-84.0,17.0
Island,269,145,45,79,-1.0,-28.0,127.0
Full compound,157,154,0,3,-21.0,-525.0,13.0


## Apply the population filter

This applies the configured edge-compound filter and produces `plot_df`, the common state-level table used below. No association screening or model-selection figures are retained here.


In [5]:
plot_df = analysis_df.loc[~analysis_df.technical_alias & analysis_df.primary_pair_complete].copy()
print(f"Plotting {len(plot_df)} states from {plot_df['example_id'].nunique()} examples.")
# No historical domain/edge-compound filter: current matched-domain metadata must first be validated.


Plotting 2172 states from 15 examples.


## Gauge hydrographs and optional model overlay

Select any or all cases. `INCLUDE_MODEL=False` gives gauges only; plot buttons also toggle model visibility. UTC is recovered from the original forcing after verifying identical time/flow arrays. No alignment to the observed downstream peak is performed. Set `NORMALIZE=False` to compare actual discharge; normalization is explicitly shape-only. Peak tables refer to the quality-sheet observation window; a model peak outside it requires inspecting the full model event.

If the model peak lies beyond the gauge review window, the display extends to include it; a dotted line marks the end of that window. Observations and peak tables retain the selected gauge window.

In [6]:
GAUGE_EXAMPLES = ["37"]  # Current case; use sorted(analysis_df.example_id.unique()) for all networks.
INCLUDE_MODEL = True
NORMALIZE = False
gauge_peak_tables = []
for example in GAUGE_EXAMPLES:
    fig, table = gauge_overlay(example, analysis=analysis_df, include_model=INCLUDE_MODEL, normalize=NORMALIZE)
    fig.show()
    gauge_peak_tables.append(table)
display(pd.concat(gauge_peak_tables, ignore_index=True))

,example_id,series,peak_utc,peak_cms
0,37,Prepared upstream gauge,2014-06-27 18:29:30+00:00,2675.658834
1,37,Prepared downstream gauge,2014-06-28 11:14:30+00:00,4699.652639
2,37,Applied forcing,2014-06-27 18:29:30+00:00,2675.658834
3,37,Current parent outlet,2014-06-28 09:15:30+00:00,2628.601639


## Parent and child network states

Case-by-case hydrograph interpretation is recorded in **`RAPID/reviews/hydrograph_match_review.xlsx`** (review dated 5 October 2026). Use this section to inspect the actual parent and child geometry.

Choose one example and optionally a subset of `(mode, state_id)` keys from the table. `UNIT_SELECTION = None` selects all eligible states; **at most `MAX_NETWORK_PAIRS` (default 6) are drawn**. Any omitted states are counted explicitly. An empty list selects none. Default browsing excludes technical aliases. Explicit `(mode, state_id)` selections may include aliases; the available-unit table labels them with `technical_alias`. The modes are `indep`, `compound`, and `island`.

Each pair highlights the selected matched domain on the current parent and child networks. Saved domain membership is checked against current topology. ZOOM_TO_DOMAIN=True zooms both maps to the shared domain extent; False shows network context. A containing/enclosing domain is labelled explicitly when applicable.


Each selected pair also has a **parent–child hydrograph** using the same example, unit selection and upper limit. These are whole-network outlet responses to the selected collapse, not local unit-outlet discharges. Blue is the parent; dashed red is the child. The shared input forcing is optional (`SHOW_INPUT_FORCING`). Set `SHOW_NETWORK_MAPS=False` to show only hydrographs, or `SHOW_HYDROGRAPHS=False` for only maps. Saved UTC timestamps, discharge magnitudes and the full routed event are preserved; use Plotly zoom to inspect overlapping peaks.


In [24]:
# Reload the shared selector as well as the current-output plotting helpers.
import importlib
import rapid_tools.hydrograph_review as _selection_helpers
import rapid_tools.current_review as _current_review
importlib.reload(_selection_helpers)
import rapid_tools.current_domains as _current_domains
importlib.reload(_current_domains)
importlib.reload(_current_review)
from rapid_tools.current_review import select_network_pairs, parent_child_network_figure, parent_child_hydrograph_figure

NETWORK_EXAMPLE = "37"
NETWORK_MODES = None  # All modes; or ["island"] to filter the table before choosing a row.
MAX_NETWORK_PAIRS = 1
ZOOM_TO_DOMAIN = True
SHOW_NETWORK_MAPS = True
SHOW_HYDROGRAPHS = True
SHOW_INPUT_FORCING = True

analysis_df = _current_review.add_centerline_comparisons(
    analysis_df, [NETWORK_EXAMPLE]
)

available_units = analysis_df.loc[
    analysis_df.example_id.eq(NETWORK_EXAMPLE),
    ["mode", "state_id", "technical_alias", "target_id", "sampled_peak_shift_minutes", "reviewed_feature_shift_minutes", "peak_total_change_pct"],
].drop_duplicates(["mode", "state_id"]).sort_values(["mode", "state_id"])
if NETWORK_MODES is not None:
    available_units = available_units[available_units["mode"].isin(NETWORK_MODES)]
print(f"{len(available_units)} available units/states; first 30 keys shown below. Full table: available_units.")
# display(available_units.head(30))

# Select the third row of the freshly built table (zero-based position 2).
# Each key needs BOTH mode and state_id, even when NETWORK_MODES is set.
selected_unit = available_units.iloc[2]
# UNIT_SELECTION = [(selected_unit["mode"], selected_unit["state_id"])]
UNIT_SELECTION = [
    ("island", "island_37_282"),
    # ("indep", "S038_unit_4"),
    # ("indep", "S130_unit_126"),
    # ("compound", "CB002_bubble_2"),
    # ("centerline", "fully_collapsed")

]
# UNIT_SELECTION =None

# Alternatively: UNIT_SELECTION = None for all eligible states, or [] for none.
selected_networks, omitted_networks = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Plotting {len(selected_networks)} pairs. Upper limit: {MAX_NETWORK_PAIRS}. Omitted by limit: {len(omitted_networks)}.")
for _, selected_network in selected_networks.iterrows():
    display(selected_network[["parent_peak_utc","child_peak_utc","sampled_peak_shift_minutes",
        "reviewed_feature_shift_minutes","peak_total_change_pct","parent_outlet_parquet","outlet_parquet"]].to_frame("current output"))
    if SHOW_NETWORK_MAPS:
        try:
            # parent_child_network_figure(selected_network, zoom_to_domain=ZOOM_TO_DOMAIN).show()
            parent_child_network_figure(selected_network, zoom_to_domain=ZOOM_TO_DOMAIN,
                                        layout='overlay', show_children=True).show()
        except (ValueError, FileNotFoundError) as exc:
            print(f"Map unavailable (hydrograph review continues): {exc}")
    if SHOW_HYDROGRAPHS:
        parent_child_hydrograph_figure(selected_network,show_title=False,
        show_forcing=False,show_downstream_gauge=False,
        show_parent=True,show_children=True,
        x_range=(pd.Timestamp("2014-06-28 04:00"), "2014-06-28 14:00"),
        y_range=(2600,2630),
        show_peaks=True).show()


653 available units/states; first 30 keys shown below. Full table: available_units.
Plotting 1 pairs. Upper limit: 1. Omitted by limit: 0.


,current output
parent_peak_utc,2014-06-28T09:15:30+00:00
child_peak_utc,2014-06-28T09:35:30+00:00
sampled_peak_shift_minutes,20.0
reviewed_feature_shift_minutes,20.0
peak_total_change_pct,-0.251233
parent_outlet_parquet,/Volumes/PhD/river_hierarchy/network_variants/...
outlet_parquet,/Volumes/PhD/river_hierarchy/network_variants/...


## Multiple collapse hydrographs in one graph

Uses the same `NETWORK_EXAMPLE`, `NETWORK_MODES`, `UNIT_SELECTION`, `MAX_NETWORK_PAIRS` and `SHOW_INPUT_FORCING` controls above. The parent is drawn once, with one line per selected collapse. `None` selects all eligible states up to the stated limit; click legend entries to isolate curves. These are whole-network outlet hydrographs on verified shared clocks, without peak alignment. The selection table preserves mode/state keys, including deliberately selected aliases; coincident lines are possible.


In [8]:
# Reload helpers so this cell also works in an already-running kernel.
import importlib
import rapid_tools.current_review as hydrograph_review
importlib.reload(hydrograph_review)
from rapid_tools.current_review import collapse_hydrograph_overlay, select_network_pairs

overlay_networks, overlay_omitted = select_network_pairs(
    analysis_df, NETWORK_EXAMPLE, units=UNIT_SELECTION,
    modes=NETWORK_MODES, max_pairs=MAX_NETWORK_PAIRS,
)
print(f"Overlay: {len(overlay_networks)} collapse states; limit {MAX_NETWORK_PAIRS}; omitted {len(overlay_omitted)}.")
display(overlay_networks[["mode", "state_id", "target_id", "hydraulic_domain_type"]])
if len(overlay_networks):
    collapse_hydrograph_overlay(overlay_networks, show_forcing=SHOW_INPUT_FORCING).show()
else:
    print("No collapse states selected.")


Overlay: 1 collapse states; limit 1; omitted 0.


,mode,state_id,target_id,hydraulic_domain_type
0,island,island_15_047,island_15_047,Current whole-network output


## Shared review helpers

These helpers define the three lag classes, compact distributions, stable review indices, and the reusable state inspector.

In [9]:
# import plotly.express as px

# # Shared state-level review frame. Timing classes are mutually exclusive and use
# # the stored percentage shift relative to the base state.
# investigation_df = plot_df.copy()
# lag_metric = "lag_to_inflow_peak_pct_shift_vs_base"
# EQ_METRIC = "Lambda_matched"
# investigation_df[lag_metric] = pd.to_numeric(investigation_df[lag_metric], errors="coerce")
# investigation_df[EQ_METRIC] = pd.to_numeric(investigation_df[EQ_METRIC], errors="coerce")
# investigation_df["neg_log10_lambda"] = -np.log10(investigation_df[EQ_METRIC].where(lambda s: s > 0))

# compact_columns = [
#     "review_index", "example_id", "mode", "state_id", "target_id", "plot_category",
#     lag_metric, EQ_METRIC, "neg_log10_lambda", "hydraulic_domain_type",
#     "parent_domain_n_links", "child_domain_n_links", "length_fraction",
#     "width_fraction_of_total",
# ]

# def _lag_subset_index_plot(frame, *, title, index_value):
#     fig = px.scatter(
#         frame, x="review_index", y=lag_metric, color="mode", symbol="plot_category",
#         hover_data=[c for c in compact_columns if c in frame.columns and c != "review_index"],
#         template="plotly_white", title=title,
#     )
#     fig.update_traces(marker=dict(size=10, line=dict(width=0.7, color="black")))
#     fig.add_hline(y=0, line_dash="dash", line_color="grey")
#     fig.add_vline(x=index_value, line_dash="dot", line_color="black")
#     fig.update_xaxes(dtick=1, title="subset index (change the REVIEW_INDEX below)")
#     fig.update_yaxes(title="lag to inflow peak shift vs base [%]")
#     return fig

# def _eq_distribution(frame, *, title):
#     fig = px.histogram(
#         frame, x=EQ_METRIC, color="mode", facet_col="plot_category",
#         barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title=title,
#         hover_data=[c for c in ["example_id", "state_id", "target_id", lag_metric] if c in frame],
#     )
#     fig.update_xaxes(type="log", title="Lambda_K (selected hydraulic domain; log scale)")
#     fig.update_yaxes(title="state count")
#     return fig



## Shared state inspector

The inspector is called by each section below. It shows the selected hydraulic domain when available; otherwise it explicitly falls back to the full-network geometry view.

In [10]:
# import geopandas as gpd
# from plotly.subplots import make_subplots
# def _add_link_layer(fig, links, *, name, color, width, opacity=1.0, row=1, col=1):
#     xs, ys = [], []
#     for geometry in links.geometry.dropna():
#         geometries = list(geometry.geoms) if geometry.geom_type == "MultiLineString" else [geometry]
#         for line in geometries:
#             x, y = line.xy
#             xs.extend(x); ys.extend(y); xs.append(None); ys.append(None)
#     if xs:
#         fig.add_trace(go.Scattergl(x=xs, y=ys, mode="lines", line=dict(color=color, width=width), opacity=opacity, name=name, hoverinfo="skip"), row=row, col=col)


# def inspect_collapsed_state(example_id, mode, state_id):
#     row=analysis_df[(analysis_df.example_id==example_id)&(analysis_df['mode']==mode)&(analysis_df.state_id==state_id)].iloc[0]
#     child_run=Path(row.refreshed_run_dir);parent_dir=base_directory(example_id)
#     def links(directory):
#         manifest=json.loads((directory/'hydraulic_context/hydraulic_context_manifest.json').read_text())
#         return gpd.read_file(manifest['input_paths']['directed_links'])
#     parent=links(parent_dir);child=links(child_run.parent)
#     pids=[int(x) for x in row.matched_base_link_ids.split(',')];cids=[int(x) for x in row.child_link_ids.split(',')]
#     fig=make_subplots(rows=1,cols=2,subplot_titles=['Refreshed comparison domain','Parent and child outlet'])
#     _add_link_layer(fig,parent,name='Parent network',color='#cbd5e1',width=1)
#     _add_link_layer(fig,parent[parent.id_link.isin(pids)],name='Parent domain',color='#2563eb',width=3)
#     _add_link_layer(fig,child[child.id_link.isin(cids)],name='Child domain',color='#dc2626',width=3)
#     for directory,name,color in [(parent_dir/'run','Parent','#2563eb'),(child_run,'Child','#dc2626')]:
#         h=pd.read_csv(directory/'outlet_hydrograph.csv');t,_=event_clock(example_id,h)
#         fig.add_trace(go.Scatter(x=t,y=h.q_outlet_cms,name=name,line=dict(color=color)),row=1,col=2)
#     fig.update_layout(title=f'{example_id} / {state_id}: {row.hydraulic_domain_type}',height=550,template='plotly_white')
#     fig.update_yaxes(scaleanchor='x',scaleratio=1,row=1,col=1)
#     fig.update_xaxes(title_text='Time (UTC)',row=1,col=2);fig.update_yaxes(title_text='Discharge (m³/s)',row=1,col=2);fig.show()
#     display(row[['Lambda_matched','domain_expanded','parent_domain_n_links','child_domain_n_links','peak_time_shift_hours','interpolated_peak_shift_hours','peak_interpolation_status','event_duration_change_pct','peak_excess_change_pct']].to_frame('value'))


## 1. Zero lag shift — timing ties

These states have exactly zero stored shift in lag to the inflow peak. The purpose is to assess their selected-domain equivalent-resistance distribution and inspect individual ties, not to treat them as delayed responses.

In [11]:
# ZERO_LAG_REVIEW_INDEX = 2
# zero_lag_cases = investigation_df.loc[investigation_df[lag_metric].eq(0)].sort_values(
#     ["example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# zero_lag_cases.insert(0, "review_index", zero_lag_cases.index)
# if zero_lag_cases.empty:
#     raise ValueError("No zero lag-shift states were found.")
# if not 0 <= ZERO_LAG_REVIEW_INDEX < len(zero_lag_cases):
#     raise IndexError(f"ZERO_LAG_REVIEW_INDEX must be between 0 and {len(zero_lag_cases) - 1}")
# zero_lag_selected = zero_lag_cases.iloc[ZERO_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"zero_lag_states": [len(zero_lag_cases)], "all_positive_or_zero": [int(investigation_df[lag_metric].ge(0).sum())]}))
# _eq_distribution(zero_lag_cases, title="Zero lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(zero_lag_cases, title=f"Zero lag shift candidates (n={len(zero_lag_cases)})", index_value=ZERO_LAG_REVIEW_INDEX).show()
# display(zero_lag_selected[[c for c in compact_columns if c in zero_lag_selected.index]].to_frame("selected zero-lag state"))
# inspect_collapsed_state(str(zero_lag_selected.example_id), str(zero_lag_selected['mode']), str(zero_lag_selected.state_id))


## 2. Positive lag shift — delayed outlet peak

These states have a positive relative lag shift. This section shows the lag distribution and selected-domain resistance-ratio distribution, then lets you inspect one state at a time.

In [12]:
# POSITIVE_LAG_REVIEW_INDEX = 0
# positive_lag_cases = investigation_df.loc[investigation_df[lag_metric].gt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# positive_lag_cases.insert(0, "review_index", positive_lag_cases.index)
# if positive_lag_cases.empty:
#     raise ValueError("No positive lag-shift states were found.")
# if not 0 <= POSITIVE_LAG_REVIEW_INDEX < len(positive_lag_cases):
#     raise IndexError(f"POSITIVE_LAG_REVIEW_INDEX must be between 0 and {len(positive_lag_cases) - 1}")
# positive_lag_selected = positive_lag_cases.iloc[POSITIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"positive_lag_states": [len(positive_lag_cases)], "lag_min_pct": [positive_lag_cases[lag_metric].min()], "lag_max_pct": [positive_lag_cases[lag_metric].max()]}))
# px.histogram(positive_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Positive lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(positive_lag_cases, title="Positive lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(positive_lag_cases, title=f"Positive lag shift candidates (n={len(positive_lag_cases)})", index_value=POSITIVE_LAG_REVIEW_INDEX).show()
# display(positive_lag_selected[[c for c in compact_columns if c in positive_lag_selected.index]].to_frame("selected positive-lag state"))
# inspect_collapsed_state(str(positive_lag_selected.example_id), str(positive_lag_selected['mode']), str(positive_lag_selected.state_id))


In [13]:
import importlib
import rapid_tools.current_review as review
importlib.reload(review)

<module 'rapid_tools.current_review' from '/Users/6256481/Code/river-hierarchy/RAPID/src/rapid_tools/current_review.py'>

## 3. Negative lag shift — earlier outlet peak

These states have a negative relative lag shift. They are kept separate because their physical interpretation differs from delayed peak timing; use the same distributions and inspector to review them.

In [14]:
# NEGATIVE_LAG_REVIEW_INDEX = 0
# negative_lag_cases = investigation_df.loc[investigation_df[lag_metric].lt(0)].sort_values(
#     [lag_metric, "example_id", "mode", "state_id"]
# ).reset_index(drop=True)
# negative_lag_cases.insert(0, "review_index", negative_lag_cases.index)
# if negative_lag_cases.empty:
#     raise ValueError("No negative lag-shift states were found.")
# if not 0 <= NEGATIVE_LAG_REVIEW_INDEX < len(negative_lag_cases):
#     raise IndexError(f"NEGATIVE_LAG_REVIEW_INDEX must be between 0 and {len(negative_lag_cases) - 1}")
# negative_lag_selected = negative_lag_cases.iloc[NEGATIVE_LAG_REVIEW_INDEX]

# display(pd.DataFrame({"negative_lag_states": [len(negative_lag_cases)], "lag_min_pct": [negative_lag_cases[lag_metric].min()], "lag_max_pct": [negative_lag_cases[lag_metric].max()]}))
# px.histogram(negative_lag_cases, x=lag_metric, color="mode", facet_col="plot_category", barmode="overlay", opacity=0.72, nbins=28, template="plotly_white", title="Negative lag shift distribution").update_xaxes(title="lag to inflow peak shift vs base [%]").update_yaxes(title="state count").show()
# _eq_distribution(negative_lag_cases, title="Negative lag shift: selected-domain Lambda_K distribution").show()
# _lag_subset_index_plot(negative_lag_cases, title=f"Negative lag shift candidates (n={len(negative_lag_cases)})", index_value=NEGATIVE_LAG_REVIEW_INDEX).show()
# display(negative_lag_selected[[c for c in compact_columns if c in negative_lag_selected.index]].to_frame("selected negative-lag state"))
# inspect_collapsed_state(str(negative_lag_selected.example_id), str(negative_lag_selected['mode']), str(negative_lag_selected.state_id))


## Peak-window ambiguity across all 16 cases — 5 October 2026

This audit leaves routing and primary metrics unchanged. Exact maxima, near-peak blocks and normalization sensitivities are retained separately. A maximum **span may contain gaps**; its midpoint is only a diagnostic. The 99/99.5% discharge windows are not confidence intervals or event-duration thresholds. Case 05’s green window is the user-proposed earlier feature, not an adopted replacement peak or proof of lateral inflow. A representative peak rule must be independent of model agreement.

Changing observed peak choices affects validation and gauge-scaled speed; changing the forcing reference time also rescales model percentage lag. Absolute parent–child timing remains unchanged when only those references change.

In [15]:
# # The Excel workbook is now the primary case-review record.
# from IPython.display import FileLink
# display(FileLink(str(REPO_ROOT / "RAPID/reviews/hydrograph_match_review.xlsx")))


## Reference ΔJ versus real-network centroid response — Manning or BASED

Set **`REAL_J_METHOD`** below, then rerun this cell:

- `'based_midpoint'`: fixed BASED depths with midpoint reference discharge, completed run `02_based_midpoint`.
- `'manning_midpoint'` (default): fixed Manning-normal depths, completed run `03_manning_midpoint`.
- `'based_original'`: fixed BASED depths with the original discharge reference, completed run `01_based_slope_geometry_correction`.

Each method has **2,185 collapse comparisons across 15 real networks per window**, excluding the artificial single-centerline control. The depth, Manning n, allocation and centroid responses come from the **same run**, with each child's own parent as reference. Roughness was calibrated separately by run and is reused across that run's parent/children; switching methods is not an isolated depth intervention. These are modelled responses on real geometries, not observations of collapsed rivers.

$$J=\sum_e f_e\frac{3nL_e}{5h_e^{2/3}\sqrt{S_e}},\qquad \Delta J=J_{\rm child}-J_{\rm parent}.$$

For **Manning-normal depths**, J is incremental steady storage at prescribed reference discharge. For **BASED depths**, the same expression evaluates the frozen Manning tangent at the prescribed BASED depth; the normal-flow relation at allocated reference discharge need not hold. Do not apply $J=3V/(5Q_{\rm in})$ to these BASED states. No simulated centroid or dynamic depth enters ΔJ.

**Celerity-limit audit:** the formula uses uncapped Manning celerity. The saved router may cap it. Hover and the outlier table report the resulting change in the parent–child mean-time descriptor, `Δrouted mean − ΔJ`. At the primary window, 13 BASED comparisons include a capped parent or child; the largest absolute descriptor difference is 1.95 min for midpoint BASED and 1.61 min for original-reference BASED. Manning has no caps. Flagged points remain included and ΔJ is not silently replaced with capped values.

**Outlier screen:** fit one ordinary least-squares line with intercept to all 2,185 comparisons within the selected method/window. Compute $z=(r-\bar r)/s_r$ from residuals around that line, with sample standard deviation (ddof=1), and flag **|z| > 3**. Diamonds mark those comparisons; circles mark the others. This is an exploratory screen, not a significance test, held-out prediction, or production model. Outliers can inflate the standard deviation; the robust alternative is deferred. Filtering the displayed network does **not** refit the line or change the pooled z-scores.

Both axes are child minus parent in minutes. Black dashed = 1:1; blue = descriptive OLS. All selected points remain; Full range/Central zoom change the view only. Set `REAL_J_WINDOW` to `'0.5%'`, `'1%'` or `'2%'`, and optionally `REAL_J_EXAMPLE`. Tables follow those selections and list every |z|>3 case ranked by |z|.

The finite-window centroid definition is unchanged. Agreement partly reflects shared routing physics and is not independent physical validation. The outlier-detail and full-record sections below remain explicitly **Manning-only**; this selector changes this overview plot and its tables, not those investigations. No routing or RF training is performed.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from IPython.display import display,HTML

REAL_J_METHOD = 'manning_midpoint'  # 'based_midpoint' / 'manning_midpoint' / 'based_original'
REAL_J_WINDOW = '0.5%'  # '0.5%', '1%', '2%'
REAL_J_EXAMPLE = 'all'  # or e.g. '22'; pooled fit/z-scores are retained
real_j_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
sys.path.insert(0,str(real_j_root/'RAPID/investigations/three_run_review_20261029'))
import incremental_storage_methods as real_j_methods
assert REAL_J_METHOD in real_j_methods.METHODS
real_j_prov=json.loads((real_j_methods.DEST/'provenance.json').read_text())
assert real_j_methods.digest(real_j_methods.__file__)==real_j_prov['script_sha256'], 'Rebuild method outputs after source edits.'
real_j_run,real_j_label=real_j_methods.METHODS[REAL_J_METHOD]
for real_j_path,real_j_hash in real_j_prov['source_sha256'].items():
    if real_j_run in Path(real_j_path).parts:
        assert real_j_methods.digest(real_j_path)==real_j_hash,f'Source changed: {real_j_path}'
real_j_data=pd.read_parquet(real_j_methods.DEST/'comparisons.parquet')
real_j_selected=real_j_data[(real_j_data.method==REAL_J_METHOD)&(real_j_data.window==REAL_J_WINDOW)].copy()
assert len(real_j_selected)==2185
real_j_scores=pd.read_csv(real_j_methods.DEST/'scores.csv')
real_j_score=real_j_scores[(real_j_scores.method==REAL_J_METHOD)&(real_j_scores.window==REAL_J_WINDOW)].iloc[0]
if REAL_J_EXAMPLE!='all':
    assert REAL_J_EXAMPLE in set(real_j_selected.example)
    real_j_selected=real_j_selected[real_j_selected.example.eq(REAL_J_EXAMPLE)].copy()
real_j_examples=sorted(real_j_selected.example.unique())
real_j_colors=dict(zip(sorted(real_j_data.example.unique()),qualitative.Dark24))
real_j_xy=real_j_selected[['delta_J_minutes','centroid_shift_minutes']].to_numpy()
real_j_pad=max(np.ptp(real_j_xy)*.04,.1)
real_j_full=[float(real_j_xy.min()-real_j_pad),float(real_j_xy.max()+real_j_pad)]
real_j_q=np.quantile(real_j_xy,[.01,.99]);real_j_pad=max(np.ptp(real_j_q)*.08,.1)
real_j_zoom=[float(min(real_j_q[0],0)-real_j_pad),float(max(real_j_q[1],0)+real_j_pad)]
real_j_fig=go.Figure()
for real_j_ex,real_j_g in real_j_selected.groupby('example'):
    real_j_flags=real_j_g.outlier_abs_z_gt3.astype(bool)
    real_j_fig.add_trace(go.Scatter(x=real_j_g.delta_J_minutes,y=real_j_g.centroid_shift_minutes,
        mode='markers',name=real_j_ex,
        marker=dict(size=np.where(real_j_flags,11,7),symbol=np.where(real_j_flags,'diamond','circle'),
                    color=real_j_colors[real_j_ex],opacity=.75),
        customdata=real_j_g[['example','state','ols_residual_minutes','residual_z','n','Qref_cms','celerity_cap_effect_minutes','capped_reaches','parent_capped_reaches']].to_numpy(),
        hovertemplate=('Network %{customdata[0]} · %{customdata[1]}'
            '<br>ΔJ %{x:.3f} min<br>Centroid %{y:.3f} min'
            '<br>OLS residual %{customdata[2]:.3f} min<br>Residual z %{customdata[3]:.3f}'
            '<br>Manning n %{customdata[4]:.5f}<br>Reference Q %{customdata[5]:.2f} m³/s'
            '<br>Capping effect on ΔJ %{customdata[6]:.3f} min'
            '<br>Capped reaches: child %{customdata[7]}, parent %{customdata[8]}<extra></extra>')))
real_j_fig.add_trace(go.Scatter(x=real_j_full,y=real_j_full,mode='lines',name='1:1 equality',
    line=dict(color='black',dash='dash',width=1.5),hoverinfo='skip'))
real_j_fig.add_trace(go.Scatter(x=real_j_full,y=[real_j_score.ols_slope*x+real_j_score.ols_intercept_minutes for x in real_j_full],
    mode='lines',name=f'Pooled OLS slope {real_j_score.ols_slope:.3f}',line=dict(color='#2874A6',width=2),hoverinfo='skip'))
real_j_count=int(real_j_selected.outlier_abs_z_gt3.astype(bool).sum())
real_j_fig.update_layout(template='plotly_white',height=800,
    title=dict(text=f'Real networks: ΔJ versus modelled centroid shift<br><sup>{real_j_label} · {REAL_J_WINDOW} window · '
        f'{len(real_j_selected):,} comparisons · {real_j_count} diamonds: |residual z| > 3</sup>',x=.04),
    xaxis=dict(title='ΔJ: child − parent (minutes)',range=real_j_full,zeroline=True,zerolinecolor='#aaa',constrain='domain'),
    yaxis=dict(title='Modelled centroid shift: child − parent (minutes)',range=real_j_full,
               scaleanchor='x',scaleratio=1,zeroline=True,zerolinecolor='#aaa',constrain='domain'),
    legend=dict(title='Network / reference lines',itemsizing='constant'),hovermode='closest',dragmode='zoom',
    margin=dict(l=85,r=40,t=145,b=75),
    updatemenus=[dict(type='buttons',direction='right',x=0,y=1.09,xanchor='left',buttons=[
        dict(label='Full range',method='relayout',args=[{'xaxis.range':real_j_full,'yaxis.range':real_j_full,'xaxis.autorange':False,'yaxis.autorange':False}]),
        dict(label='Central zoom',method='relayout',args=[{'xaxis.range':real_j_zoom,'yaxis.range':real_j_zoom,'xaxis.autorange':False,'yaxis.autorange':False}])])])
real_j_html_path=real_j_methods.DEST/f'delta_J_{REAL_J_METHOD}_{REAL_J_WINDOW.replace("%","pct")}_{REAL_J_EXAMPLE}.html'
real_j_fig.write_html(real_j_html_path,include_plotlyjs=True,config={'scrollZoom':True,'displaylogo':False})
display(HTML(real_j_fig.to_html(full_html=False,include_plotlyjs=True,config={'scrollZoom':True,'displaylogo':False})))
assert sum(len(trace.x) for trace in real_j_fig.data[:-2])==len(real_j_selected)
display(real_j_scores[(real_j_scores.method==REAL_J_METHOD)&(real_j_scores.window==REAL_J_WINDOW)].round(4))
real_j_outliers=real_j_selected[real_j_selected.outlier_abs_z_gt3.astype(bool)].copy()
real_j_outliers=real_j_outliers.loc[real_j_outliers.residual_z.abs().sort_values(ascending=False).index]
display(real_j_outliers[['example','state','delta_J_minutes','centroid_shift_minutes','ols_residual_minutes','residual_z','celerity_cap_effect_minutes']].round(4))
print('Outliers use pooled OLS residuals for this method/window; the identity-line error is not used for z-scores.')
print('Selected-network filtering does not refit the pooled line. No routing or predictive-model training.')


,method,window,n,networks,ols_slope,ols_intercept_minutes,ols_r2,residual_sd_minutes,ols_mae_minutes,outliers_abs_z_gt3,identity_r2,identity_mae_minutes,comparisons_with_capped_reaches,max_abs_cap_effect_minutes
0,manning_midpoint,0.5%,2185,15,0.7288,-0.1621,0.9867,1.4166,0.3977,25,0.8494,0.8713,0,0.0


,example,state,delta_J_minutes,centroid_shift_minutes,ols_residual_minutes,residual_z,celerity_cap_effect_minutes
6062,55,all_closed_units_collapsed,-192.9710,-170.0561,-29.2556,-20.6513,0.0
3084,37,island_37_282,22.3771,-9.5872,-25.7337,-18.1652,0.0
4524,45,all_closed_units_collapsed,-124.1600,-112.5230,-21.8724,-15.4395,0.0
4936,46,all_closed_units_collapsed,-531.9648,-369.2871,18.5742,13.1114,0.0
2780,37,S082_unit_324,17.5498,-5.5602,-18.1885,-12.8391,0.0
4066,41,all_closed_units_collapsed,-275.7563,-215.8847,-14.7497,-10.4117,0.0
2890,37,S192_unit_38,12.7609,-1.0153,-10.1534,-7.1672,0.0
6510,56,all_closed_units_collapsed,-49.6110,-45.7053,-9.3864,-6.6258,0.0
3002,37,S304_unit_190,8.6837,-2.8860,-9.0526,-6.3902,0.0
282,05,all_closed_units_collapsed,-173.3498,-118.2052,8.2953,5.8556,-0.0


Outliers use pooled OLS residuals for this method/window; the identity-line error is not used for z-scores.
Selected-network filtering does not refit the pooled line. No routing or predictive-model training.


## Manning ΔJ outliers — ordinary residual z-score

**25 of 2,185 collapse comparisons (1.14%)** are flagged in the primary **0.5% window**, using **|z| > 3**. These are comparisons, not individual reaches. All points participate in one pooled ordinary least-squares regression with an intercept; there are no separate network regressions and no outlier removal before fitting.

$$\hat y=0.728806\,\Delta J-0.162095\ \mathrm{min},\qquad
r=y-\hat y,\qquad z=\frac{r-\bar r}{s_r}.$$
For the primary window, the sample residual standard deviation is **1.41665 min**, so the screen corresponds to a departure of approximately **4.25 min** from the fitted line. This is an exploratory screen, not a significance test: residuals can be non-normal, outliers inflate the standard deviation, and collapses share parent networks. Robust screening is deferred.

**One Plotly outlier plot replaces the previous flattening and sign-reversal plots.** Horizontal axis = ΔJ; vertical axis = residual z-score around the fitted relationship, **not the error from the 1:1 line**. Gray points are unflagged; coloured points are flagged and grouped by network. Dashed horizontal lines mark ±3. Positive z means the measured response is later than the fitted relationship suggests; negative z means earlier. Hover identifies each state. Zoom controls only change the view; no cases are discarded.

Set `OUTLIER_WINDOW` below if desired. The numerical summary and ranked table update for that window. Whole-network `all_closed_units_collapsed` comparisons remain included and are labelled separately from local edits. The original two-method overview remains above; no routing, RF training, or production metric changes are involved.


### Separate magnitude departures from direction reversals

“Same/opposite sign” compares **ΔJ with the measured centroid shift**, not with the sign of the residual. The original pooled regression and |z|>3 selection are unchanged.

At 0.5%: **15 same-sign outliers** (12 advances, 3 delays) and **10 opposite-sign outliers** (all ΔJ positive / measured centroid negative). Circles indicate same-sign comparisons, diamonds opposite-sign; open markers flag either quantity within ±1 min of zero. That near-zero band is a descriptive sensitivity flag, not a confidence interval.

Nine same-sign outliers are `all_closed_units_collapsed` configurations: **9/15** such configurations are flagged. Other same-sign outliers are four compound members and two full compounds. All ten opposite-sign outliers are local edits: eight compound members and two islands. They occur in networks 03 (1), 37 (6), 46 (2), and 55 (1). All ten remain opposite-sign and |z|>3 in each saved window. Network 46 / `S093_unit_64` has a small −0.369 min response; the other nine have magnitudes >1 min on both axes.

There are no simple-unit outliers (**0/258**). These are descriptive concentrations, not independent statistical tests. Whole-network changes can be flagged partly because their large ΔJ magnifies departures from the pooled slope; that should be distinguished from a local sign reversal. The groups are not the earlier >1-minute sign-reversal selection: they are subdivisions of the current 25 residual-z outliers.


In [31]:
(5/10)**(3/5), (10/20)**(3/5), (7/14)**(3/5)

(0.6597539553864471, 0.6597539553864471, 0.6597539553864471)

In [29]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from IPython.display import display,HTML

OUTLIER_WINDOW='0.5%'  # '0.5%', '1%', '2%'; always fixed Manning midpoint
outlier_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
sys.path.insert(0,str(outlier_root/'RAPID/investigations/three_run_review_20261029'))
import incremental_storage_methods as oj
outlier_prov=json.loads((oj.DEST/'provenance.json').read_text())
assert oj.digest(oj.__file__)==outlier_prov['script_sha256']
for outlier_path,outlier_hash in outlier_prov['source_sha256'].items():
    if '03_manning_midpoint' in Path(outlier_path).parts:
        assert oj.digest(outlier_path)==outlier_hash,outlier_path
outlier_data=pd.read_parquet(oj.DEST/'comparisons.parquet')
outlier_data=outlier_data[(outlier_data.method=='manning_midpoint')&(outlier_data.window==OUTLIER_WINDOW)].copy()
assert len(outlier_data)==2185
outlier_labels=pd.read_parquet(outlier_root/'RAPID/outputs/uncapped_event_window_20261020/validation_group_inventory_20261023/records.parquet')[['example','state','edit_type']]
outlier_data=outlier_data.merge(outlier_labels,on=['example','state'],how='left',validate='one_to_one')
outlier_data.loc[outlier_data.state.eq('all_closed_units_collapsed'),'edit_type']='All closed units'
assert outlier_data.edit_type.notna().all()
# Verify the stored z-score definition directly; no refitting or screen-based exclusion.
outlier_res=outlier_data.ols_residual_minutes
np.testing.assert_allclose(outlier_data.residual_z,(outlier_res-outlier_res.mean())/outlier_res.std(ddof=1),rtol=1e-10,atol=1e-10)
outlier_data['sign_group']=np.select(
    [outlier_data.delta_J_minutes*outlier_data.centroid_shift_minutes<0,
     outlier_data.delta_J_minutes*outlier_data.centroid_shift_minutes>0],
    ['Opposite sign','Same sign'],default='Zero on an axis')
outlier_data['near_zero_1min']=(outlier_data.delta_J_minutes.abs()<=1)|(outlier_data.centroid_shift_minutes.abs()<=1)
outlier_flags=outlier_data.residual_z.abs()>3
assert np.array_equal(outlier_flags.to_numpy(),outlier_data.outlier_abs_z_gt3.astype(bool).to_numpy())
outlier_ranked=outlier_data[outlier_flags].copy()
outlier_ranked=outlier_ranked.loc[outlier_ranked.residual_z.abs().sort_values(ascending=False).index]
outlier_colors=dict(zip(sorted(outlier_data.example.unique()),qualitative.Dark24))
outlier_fig=go.Figure()
def outlier_trace(g,name,color,size,opacity):
    return go.Scatter(x=g.delta_J_minutes,y=g.residual_z,mode='markers',name=name,
        marker=dict(size=size,color=color,opacity=opacity,
            symbol=[('diamond' if group=='Opposite sign' else 'circle')+('-open' if near else '')
                    for group,near in zip(g.sign_group,g.near_zero_1min)]),
        customdata=g[['example','state','edit_type','centroid_shift_minutes','ols_fitted_minutes','ols_residual_minutes','sign_group','near_zero_1min']].to_numpy(),
        hovertemplate=('Network %{customdata[0]} · %{customdata[1]}<br>%{customdata[2]}'
            '<br>ΔJ %{x:.3f} min<br>Centroid %{customdata[3]:.3f} min'
            '<br>Fitted centroid %{customdata[4]:.3f} min<br>Residual %{customdata[5]:.3f} min'
            '<br>Residual z %{y:.3f}<br>%{customdata[6]}<br>Within ±1 min of zero: %{customdata[7]}<extra></extra>'))
outlier_fig.add_trace(outlier_trace(outlier_data[~outlier_flags],f'Within ±3 ({(~outlier_flags).sum():,})','#aeb6bf',5,.35))
for outlier_ex,outlier_group in outlier_ranked.groupby('example'):
    outlier_fig.add_trace(outlier_trace(outlier_group,f'{outlier_ex} ({len(outlier_group)})',outlier_colors[outlier_ex],10,.9))
for threshold in [-3,3]:outlier_fig.add_hline(y=threshold,line_dash='dash',line_color='#b03a2e',annotation_text=f'z = {threshold:+d}')
outlier_fig.add_hline(y=0,line_color='#777',line_width=1)
outlier_x=outlier_data.delta_J_minutes.to_numpy();outlier_pad=max(np.ptp(outlier_x)*.04,1)
outlier_range=[float(outlier_x.min()-outlier_pad),float(outlier_x.max()+outlier_pad)]
outlier_fig.update_layout(template='plotly_white',height=760,
    title=f'Manning ΔJ outliers · {OUTLIER_WINDOW} window<br><sup>{len(outlier_ranked)} / {len(outlier_data):,} comparisons with |z| > 3; colours identify networks</sup>',
    xaxis=dict(title='ΔJ: child − parent (minutes)',range=outlier_range),
    yaxis=dict(title='Standardized residual from pooled OLS line (z-score)'),
    legend=dict(title='Flagged network (count)'),hovermode='closest',margin=dict(t=140),
    updatemenus=[dict(type='buttons',direction='right',x=0,y=1.1,buttons=[
        dict(label='Full ΔJ range',method='relayout',args=[{'xaxis.range':outlier_range,'xaxis.autorange':False}]),
        dict(label='Zoom: −30 to +30 min',method='relayout',args=[{'xaxis.range':[-30,30],'xaxis.autorange':False}])])])
outlier_dest=oj.DEST/'manning_outliers';outlier_dest.mkdir(exist_ok=True)
outlier_fig.write_html(outlier_dest/f'outliers_{OUTLIER_WINDOW.replace("%","pct")}.html',include_plotlyjs=True)
display(HTML(outlier_fig.to_html(full_html=False,include_plotlyjs=True,config={'scrollZoom':True,'displaylogo':False})))
assert sum(len(t.x) for t in outlier_fig.data)==len(outlier_data)
display(pd.DataFrame([dict(window=OUTLIER_WINDOW,comparisons=len(outlier_data),outliers=len(outlier_ranked),
    percent=100*len(outlier_ranked)/len(outlier_data),networks_with_outliers=outlier_ranked.example.nunique(),
    residual_SD_min=outlier_res.std(ddof=1),three_SD_min=3*outlier_res.std(ddof=1))]).round(4))
outlier_table=outlier_ranked[['example','state','edit_type','sign_group','near_zero_1min','delta_J_minutes','centroid_shift_minutes','ols_fitted_minutes','ols_residual_minutes','residual_z']].copy()
outlier_table.insert(0,'rank',np.arange(1,len(outlier_table)+1))
outlier_table.to_csv(outlier_dest/f'ranked_{OUTLIER_WINDOW.replace("%","pct")}.csv',index=False)
outlier_group_summary=pd.crosstab(outlier_ranked.sign_group,outlier_ranked.edit_type).reindex(
    columns=['All closed units','Compound member','Full compound','Island','Simple'],fill_value=0)
outlier_group_summary['Total']=outlier_group_summary.sum(axis=1)
display(outlier_group_summary)
outlier_rates=outlier_data.assign(flag=outlier_flags).groupby('edit_type').agg(comparisons=('flag','size'),outliers=('flag','sum'))
outlier_rates['outlier_pct']=100*outlier_rates.outliers/outlier_rates.comparisons
display(outlier_rates.round(3))
for outlier_sign in ['Same sign','Opposite sign']:
    display(HTML(f'<h4>{outlier_sign} outliers</h4>'))
    display(outlier_table[outlier_table.sign_group.eq(outlier_sign)].round(3))
outlier_group_summary.to_csv(outlier_dest/f'sign_groups_{OUTLIER_WINDOW.replace("%","pct")}.csv')
outlier_rates.to_csv(outlier_dest/f'edit_type_rates_{OUTLIER_WINDOW.replace("%","pct")}.csv')
# Follow the selected opposite-sign cases across all windows without changing membership.
outlier_all=pd.read_parquet(oj.DEST/'comparisons.parquet')
outlier_other_windows=outlier_all[outlier_all.method.eq('manning_midpoint')].merge(
    outlier_ranked[outlier_ranked.sign_group.eq('Opposite sign')][['example','state']],
    on=['example','state'],validate='many_to_one')
outlier_other_windows[['example','state','window','delta_J_minutes','centroid_shift_minutes','residual_z','outlier_abs_z_gt3']].to_csv(
    outlier_dest/f'opposite_sign_window_check_{OUTLIER_WINDOW.replace("%","pct")}.csv',index=False)



,window,comparisons,outliers,percent,networks_with_outliers,residual_SD_min,three_SD_min
0,0.5%,2185,25,1.1442,13,1.4166,4.2499


edit_type,All closed units,Compound member,Full compound,Island,Simple,Total
sign_group,,,,,,
Opposite sign,0,8,0,2,0,10
Same sign,9,4,2,0,0,15


,comparisons,outliers,outlier_pct
edit_type,,,
All closed units,15,9,60.000
Compound member,1467,12,0.818
Full compound,177,2,1.130
Island,268,2,0.746
Simple,258,0,0.000


,rank,example,state,edit_type,sign_group,near_zero_1min,delta_J_minutes,centroid_shift_minutes,ols_fitted_minutes,ols_residual_minutes,residual_z
2162,1,55,all_closed_units_collapsed,All closed units,Same sign,False,-192.971,-170.056,-140.801,-29.256,-20.651
1570,3,45,all_closed_units_collapsed,All closed units,Same sign,False,-124.160,-112.523,-90.651,-21.872,-15.440
1794,4,46,all_closed_units_collapsed,All closed units,Same sign,False,-531.965,-369.287,-387.861,18.574,13.111
1476,6,41,all_closed_units_collapsed,All closed units,Same sign,False,-275.756,-215.885,-201.135,-14.750,-10.412
2184,8,56,all_closed_units_collapsed,All closed units,Same sign,False,-49.611,-45.705,-36.319,-9.386,-6.626
128,10,05,all_closed_units_collapsed,All closed units,Same sign,False,-173.350,-118.205,-126.500,8.295,5.856
1949,11,54,all_closed_units_collapsed,All closed units,Same sign,False,-67.965,-41.495,-49.696,8.200,5.788
1571,12,46,CB001_bubble_90,Full compound,Same sign,False,-112.683,-74.895,-82.286,7.391,5.217
898,14,22,all_closed_units_collapsed,All closed units,Same sign,False,-147.155,-114.559,-107.409,-7.150,-5.047
190,16,13,S054_unit_53,Compound member,Same sign,False,17.744,6.006,12.770,-6.764,-4.774


,rank,example,state,edit_type,sign_group,near_zero_1min,delta_J_minutes,centroid_shift_minutes,ols_fitted_minutes,ols_residual_minutes,residual_z
1286,2,37,island_37_282,Island,Opposite sign,False,22.377,-9.587,16.146,-25.734,-18.165
982,5,37,S082_unit_324,Compound member,Opposite sign,False,17.550,-5.560,12.628,-18.188,-12.839
1092,7,37,S192_unit_38,Compound member,Opposite sign,False,12.761,-1.015,9.138,-10.153,-7.167
1204,9,37,S304_unit_190,Compound member,Opposite sign,False,8.684,-2.886,6.167,-9.053,-6.390
2025,13,55,S049_unit_137,Compound member,Opposite sign,False,7.040,-2.185,4.969,-7.154,-5.050
67,15,03,S027_unit_24,Compound member,Opposite sign,False,6.553,-2.338,4.614,-6.952,-4.907
1129,18,37,S229_unit_237,Compound member,Opposite sign,False,7.109,-1.305,5.019,-6.324,-4.464
1639,22,46,S039_unit_18,Compound member,Opposite sign,False,5.066,-1.640,3.530,-5.170,-3.650
1252,23,37,island_37_119,Island,Opposite sign,False,5.113,-1.508,3.564,-5.073,-3.581
1693,25,46,S093_unit_64,Compound member,Opposite sign,True,6.031,-0.369,4.234,-4.602,-3.249


## Full saved hydrograph versus the existing event window — network 37

The real-case literature within this repository uses “full centroid” alongside a **0.5% window** (e.g. `six_panel_geometry_depth_resistance.py`), contrasting with capped hydrograph metrics. It does not mean integration over every saved simulation sample. The current real-run metric code explicitly integrates between fixed onset/end times. Synthetic full excess-event centroids have a different integration contract.

This diagnostic **removes the analysis-window cutoff**, retains the same baseline (1469.833 m³/s), and integrates from the first to the last saved sample. It also moves the endpoint through the existing saved tail. Nothing is rerouted, extrapolated, clipped, or changed in the production metrics. The “full saved” record still has an endpoint: 6 July 2014 at 15:29:30 UTC, and its final discharge is about 1806.99 m³/s, above the baseline. Thus this is not an infinite, isolated pulse returning to zero excess.

The upper panel shows the saved hydrographs; shading marks the existing event window. The lower panel shows how the measured parent–child centroid shift changes as the endpoint is extended. Solid curves use the full saved start and increasing endpoints. Dashed horizontal lines are ΔJ, not fits. The last endpoint uses the complete saved record. “Total discharge” in the table is a separate sensitivity that includes baseflow; it is not substituted for the excess-flow metric.

For `island_37_275`, the excess centroid shift moves **+9.05 → +7.43 min**; for `island_37_282`, **−9.59 → −4.00 min**. Extending the record therefore does not automatically recover ΔJ. Continuing above-baseline plateau water changes the quantity whose centroid is measured. These two examples are diagnostics, not a new all-network metric definition.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display,HTML
full_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
sys.path.insert(0,str(full_root/'RAPID/investigations/three_run_review_20261029'))
import incremental_storage as fj
from incremental_storage_patterns import moment
full_cases=['island_37_275','island_37_282']
full_data=pd.read_parquet(fj.OUT/'comparisons.parquet')
full_sources={}
def full_load(state):
    path=fj.SOURCE/'states'/'37'/state/'outlet.parquet'
    receipt=json.loads((path.parent/'completion.json').read_text())
    sha=fj.digest(path);assert sha==receipt['files'][str(path)];full_sources[str(path)]=sha
    return pd.read_parquet(path)
full_parent=full_load('S000_base')
full_t=full_parent.time_utc.astype('int64').to_numpy()/1e9
full_q=full_parent.q_outlet_cms.to_numpy()
full_row=full_data[(full_data.example=='37')&(full_data.state==full_cases[0])&(full_data.window=='0.5%')].iloc[0]
full_a=pd.Timestamp(full_row.onset_utc).timestamp();full_b=pd.Timestamp(full_row.end_utc).timestamp();full_base=full_row.baseline_cms
full_fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.13,
    subplot_titles=('Saved hydrographs; shaded area = current analysis window','Centroid shift as the integration endpoint moves through the saved tail'))
full_fig.add_trace(go.Scatter(x=full_parent.time_utc,y=full_q,name='Parent',line=dict(color='black',width=2)),row=1,col=1)
full_colors=['#0072B2','#CC79A7'];full_records=[];full_curve=[]
for full_state,full_color in zip(full_cases,full_colors):
    child=full_load(full_state);assert child.time_utc.equals(full_parent.time_utc)
    cq=child.q_outlet_cms.to_numpy()
    row=full_data[(full_data.example=='37')&(full_data.state==full_state)&(full_data.window=='0.5%')].iloc[0]
    full_fig.add_trace(go.Scatter(x=child.time_utc,y=cq,name=full_state,line=dict(color=full_color,width=1.5)),row=1,col=1)
    def shifted_centroid(start,end,baseline):
        tt=np.r_[start,full_t[(full_t>start)&(full_t<end)],end]
        mp,_=moment(tt,np.interp(tt,full_t,full_q)-baseline)
        mc,_=moment(tt,np.interp(tt,full_t,cq)-baseline)
        return (mc-mp)/60
    original=shifted_centroid(full_a,full_b,full_base)
    np.testing.assert_allclose(original,row.centroid_shift_minutes,atol=1e-5)
    for label,start,end in [('Current window',full_a,full_b),('Full start, current end',full_t[0],full_b),
        ('Current end +24 h',full_t[0],full_b+86400),('Current end +72 h',full_t[0],full_b+3*86400),
        ('Entire saved record',full_t[0],full_t[-1])]:
        full_records.append(dict(state=full_state,measurement=label,end_utc=pd.Timestamp(end,unit='s',tz='UTC'),
            delta_J_min=row.delta_J_minutes,excess_centroid_shift_min=shifted_centroid(start,end,full_base),
            total_discharge_centroid_shift_min=shifted_centroid(start,end,0)))
    ends=np.r_[np.arange(full_b,full_t[-1],3600),full_t[-1]]
    vals=[shifted_centroid(full_t[0],end,full_base) for end in ends]
    for end,val in zip(ends,vals):full_curve.append(dict(state=full_state,end_utc=pd.Timestamp(end,unit='s',tz='UTC'),centroid_shift_min=val))
    full_fig.add_trace(go.Scatter(x=pd.to_datetime(ends,unit='s',utc=True),y=vals,name=full_state+' centroid',
        line=dict(color=full_color),hovertemplate='%{x}<br>Shift %{y:.3f} min<extra>'+full_state+'</extra>'),row=2,col=1)
    full_fig.add_trace(go.Scatter(x=pd.to_datetime([full_b,full_t[-1]],unit='s',utc=True),y=[row.delta_J_minutes]*2,
        name=full_state+' ΔJ',line=dict(color=full_color,dash='dash')),row=2,col=1)
full_fig.add_hline(y=full_base,line_dash='dot',line_color='gray',row=1,col=1)
full_fig.add_vrect(x0=pd.Timestamp(full_a,unit='s',tz='UTC'),x1=pd.Timestamp(full_b,unit='s',tz='UTC'),fillcolor='gray',opacity=.12,line_width=0,row=1,col=1)
full_fig.add_hline(y=0,line_color='gray',line_width=1,row=2,col=1)
full_fig.update_yaxes(title_text='Discharge (m³/s)',row=1,col=1)
full_fig.update_yaxes(title_text='Child − parent (minutes)',row=2,col=1)
full_fig.update_xaxes(title_text='UTC; lower panel x = measurement endpoint',row=2,col=1)
full_fig.update_layout(template='plotly_white',height=850,title='Network 37: what changes when we use the entire saved hydrograph?',legend=dict(orientation='h',y=-.13),margin=dict(b=150))
full_table=pd.DataFrame(full_records)
full_out=fj.OUT/'full_record_example';full_out.mkdir(exist_ok=True)
full_table.to_csv(full_out/'endpoint_comparison.csv',index=False)
pd.DataFrame(full_curve).to_csv(full_out/'endpoint_curves.csv',index=False)
(full_out/'provenance.json').write_text(json.dumps(dict(source_sha256=full_sources,baseline_cms=full_base,
    scope='Saved record only; no extrapolation or rerouting. Existing baseline retained. No production metric changes.'),indent=2)+'\n')
full_fig.write_html(full_out/'full_saved_hydrographs.html',include_plotlyjs=True)
display(HTML(full_fig.to_html(full_html=False,include_plotlyjs=True,config={'scrollZoom':True,'displaylogo':False})))
display(full_table.round(4))


,state,measurement,end_utc,delta_J_min,excess_centroid_shift_min,total_discharge_centroid_shift_min
0,island_37_275,Current window,2014-07-01 15:21:30+00:00,12.3505,9.0542,2.5382
1,island_37_275,"Full start, current end",2014-07-01 15:21:30+00:00,12.3505,9.0542,2.0319
2,island_37_275,Current end +24 h,2014-07-02 15:21:30+00:00,12.3505,8.5437,1.9118
3,island_37_275,Current end +72 h,2014-07-04 15:21:30+00:00,12.3505,7.8593,1.7480
4,island_37_275,Entire saved record,2014-07-06 15:29:30+00:00,12.3505,7.4327,1.6417
5,island_37_282,Current window,2014-07-01 15:21:30+00:00,22.3771,-9.5872,-2.8226
6,island_37_282,"Full start, current end",2014-07-01 15:21:30+00:00,22.3771,-9.5872,-2.6864
7,island_37_282,Current end +24 h,2014-07-02 15:21:30+00:00,22.3771,-8.1789,-2.1730
8,island_37_282,Current end +72 h,2014-07-04 15:21:30+00:00,22.3771,-5.8468,-1.4238
9,island_37_282,Entire saved record,2014-07-06 15:29:30+00:00,22.3771,-4.0047,-0.9027


## Manning ΔJ outliers — three-group hydraulic investigation

The original **25 flags and pooled regression are unchanged**. This section uses completed fixed-Manning runs and analytic transfer moments; it launches no routing.

- **G1 (11):** nine all-closed-unit collapses plus two full compounds. All advance and reduce transfer spread; their proportional response differs from the pooled line.
- **G2 (4):** three delayed members increase dispersion; 46/S178_unit_44 instead resembles the aggregate advances.
- **G3 (10):** eight members and two islands. All increase transfer spread (1.24–5.47×); within-reach dispersion explains at least 99.88% of the added variance. Seven of the eight members have active slope-floor reaches. This is a model mechanism and an input-representation concern, not yet proof of a real-river effect.

**Transfer spread is not observed event width.** It includes potentially small, long-lived model tails. Matched non-outliers sometimes also increase spread. No opposite-sign case has direct saved time/grid refinement coverage.

The first plot places the existing residual z-scores against the change in analytic transfer spread. The dropdown below shows **actual saved parent and child curves**, followed by their difference. Grey marks the original response window; dotted lines mark the two window centroids. Double-click to view the full saved duration. For compact display, each ten-minute bin retains its first sample and both extrema; metrics use the original full-resolution records. Nothing is artificially shifted.

Full derivation, all 25 case notes, controls, source provenance and limitations: `RAPID/outputs/uncapped_event_window_20261020/three_run_review_20261029/incremental_storage/group_investigation/REPORT.md`.


In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
from IPython.display import HTML, display
J_GROUP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "RAPID/investigations/three_run_review_20261029").is_dir())
sys.path.insert(0, str(J_GROUP_ROOT / "RAPID/investigations/three_run_review_20261029"))
import plot_j_outlier_groups as j_group_plots
importlib.reload(j_group_plots)
J_GROUP_OUTPUT = j_group_plots.DEST
j_group_outliers = pd.read_csv(J_GROUP_OUTPUT / "outlier_diagnostics.csv", dtype={"example": str})
display(HTML(j_group_plots.overview().to_html(full_html=False, include_plotlyjs=True)))
display(j_group_outliers[["group", "example", "state", "delta_J_minutes", "centroid_shift_minutes", "residual_z", "sd_ratio", "checks"]].round(3))


,group,example,state,delta_J_minutes,centroid_shift_minutes,residual_z,sd_ratio,checks
0,G2 same-sign member,02,S016_unit_7,9.346,1.618,-3.551,1.551,2
1,G3 opposite-sign local,03,S027_unit_24,6.553,-2.338,-4.907,2.422,0
2,G1 same-sign aggregate,05,all_closed_units_collapsed,-173.350,-118.205,5.856,0.973,2
3,G2 same-sign member,13,S054_unit_53,17.744,6.006,-4.774,1.152,2
4,G1 same-sign aggregate,15,all_closed_units_collapsed,-110.601,-87.498,-4.750,0.481,2
5,G1 same-sign aggregate,22,all_closed_units_collapsed,-147.155,-114.559,-5.047,0.943,2
6,G3 opposite-sign local,37,S082_unit_324,17.550,-5.560,-12.839,4.468,0
7,G3 opposite-sign local,37,S192_unit_38,12.761,-1.015,-7.167,3.516,0
8,G3 opposite-sign local,37,S229_unit_237,7.109,-1.305,-4.464,2.774,0
9,G3 opposite-sign local,37,S304_unit_190,8.684,-2.886,-6.390,3.169,0


In [2]:
# Dropdown contains all 25 cases. Curves come directly from completed outlet.parquet records.
display(HTML(j_group_plots.wave_inspector().to_html(full_html=False, include_plotlyjs=True)))
